<a href="https://colab.research.google.com/github/MeerMusabih/FlyRank-AI-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeerMusabih/FlyRank-AI-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The Random Forest produces a probability that a page belongs to the observed declining class. Rather than treating this probability as an automatic decision, it is used to rank pages for editorial review.

Each page is assigned a recommended action based on its model score. Higher-scoring pages are placed earlier in the review queue because they more closely resemble the observed declining pages in the training data.

The recommended actions are intended for decision support. They help prioritize limited editorial resources but do not replace human judgment.

Reason codes provide a short explanation for why a page appears in the queue. They summarize observed characteristics using available features from the dataset and are intended to improve transparency rather than explain the internal behavior of the Random Forest.

In [13]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

TARGET = "is_declining_label"
df[TARGET] = (df["trend_direction"] == "down").astype(int)

ID_COLUMNS = [
    "content_id",
    "client_id"
]

LEAKAGE_COLUMNS = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

NON_FEATURE_COLUMNS = [
    "provider_used",
    "model_used"
]

excluded_columns = (
    [TARGET]
    + ID_COLUMNS
    + LEAKAGE_COLUMNS
    + NON_FEATURE_COLUMNS
)

feature_columns = [
    c for c in df.columns
    if c not in excluded_columns
]

X = df[feature_columns]
y = df[TARGET]
groups = df["client_id"]

print("Number of features:", len(feature_columns))
print(feature_columns)

Number of features: 32
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']


In [14]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(train_idx))
print("Test rows:", len(test_idx))

Training rows: 23837
Test rows: 6163


In [15]:
categorical_columns = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_columns = X_train.select_dtypes(
    include=np.number
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_columns,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore"))
                ]
            ),
            categorical_columns,
        ),
    ]
)

In [16]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            ),
        ),
    ]
)

rf_pipeline.fit(X_train, y_train)

model_score = rf_pipeline.predict_proba(X_test)[:, 1]

In [17]:
action_queue = df.iloc[test_idx][["content_id"]].copy()

action_queue["model_score"] = model_score

action_queue["recommended_action"] = np.select(
    [
        action_queue["model_score"] >= 0.80,
        action_queue["model_score"] >= 0.60,
        action_queue["model_score"] >= 0.40,
    ],
    [
        "Prioritize refresh review",
        "Editorial review",
        "Monitor",
    ],
    default="No immediate action",
)

action_queue["human_review_required"] = "Yes"

In [18]:
lookup = df.loc[
    test_idx,
    [
        "content_id",
        "days_since_last_update",
        "content_age_days",
        "avg_position",
        "impressions_90d",
    ],
]

action_queue = action_queue.merge(lookup, on="content_id")

reason_codes = []

for _, row in action_queue.iterrows():
    reasons = []

    if row["days_since_last_update"] >= 180:
        reasons.append("STALE_CONTENT")

    if row["content_age_days"] >= 270:
        reasons.append("OLDER_CONTENT")

    if row["avg_position"] > 10:
        reasons.append("LOW_SEARCH_POSITION")

    if row["impressions_90d"] >= 500:
        reasons.append("HIGH_SEARCH_VISIBILITY")

    if not reasons:
        reasons.append("MIXED_SIGNALS")

    reason_codes.append(", ".join(reasons))

action_queue["reason_codes"] = reason_codes

In [19]:
action_queue = action_queue.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

action_queue[
    [
        "content_id",
        "model_score",
        "recommended_action",
        "reason_codes",
        "human_review_required",
    ]
].head(20)

,content_id,model_score,recommended_action,reason_codes,human_review_required
0,content_2ba626fea4d6,0.990,Prioritize refresh review,OLDER_CONTENT,Yes
1,content_e988c1699454,0.975,Prioritize refresh review,"OLDER_CONTENT, LOW_SEARCH_POSITION, HIGH_SEARC...",Yes
2,content_7766ffacdcfa,0.970,Prioritize refresh review,"OLDER_CONTENT, HIGH_SEARCH_VISIBILITY",Yes
3,content_9ac61c04930e,0.955,Prioritize refresh review,"OLDER_CONTENT, HIGH_SEARCH_VISIBILITY",Yes
4,content_790cc62d4743,0.955,Prioritize refresh review,"OLDER_CONTENT, LOW_SEARCH_POSITION, HIGH_SEARC...",Yes
5,content_b23d650634c6,0.950,Prioritize refresh review,MIXED_SIGNALS,Yes
6,content_569475b335ab,0.950,Prioritize refresh review,"OLDER_CONTENT, LOW_SEARCH_POSITION, HIGH_SEARC...",Yes
7,content_0d7f4fecb391,0.940,Prioritize refresh review,"OLDER_CONTENT, LOW_SEARCH_POSITION, HIGH_SEARC...",Yes
8,content_500bd3907331,0.940,Prioritize refresh review,HIGH_SEARCH_VISIBILITY,Yes
9,content_53285c7434ac,0.935,Prioritize refresh review,MIXED_SIGNALS,Yes


## 2. Intended use and limits

### Intended use

This ranked action queue is intended for SEO and content teams that need to prioritize which pages should be reviewed for possible content refresh. The model ranks pages according to how closely they resemble the observed declining pages in the training data.

The output is designed to support editorial prioritization rather than automate content decisions. Higher-ranked pages should be reviewed alongside business goals, content quality, and subject-matter expertise before any action is taken.

### Limits

The model was trained and evaluated using observed historical data from the provided dataset. Its predictions should not be interpreted as proof that a page will decline in the future or that refreshing a page will improve its performance.

The evaluation used a client-grouped train/test split, which measures generalization to unseen clients. However, the dataset does not include observation timestamps, so the model was not validated on future time periods.

The reason codes summarize observed page characteristics for transparency. They are not explanations of how the Random Forest makes individual predictions and should not be interpreted as causal evidence.

## 3. Human review + the no-go list
### Human review

Before acting on a recommendation, a reviewer should confirm that the page is still relevant to current business goals and audience needs. The reviewer should also assess the content quality, factual accuracy, search intent, and whether the page has already been updated recently or is scheduled for revision.

The model score should be considered together with editorial judgment and any additional business information that is not available in the dataset.

### No-go list

The following decisions should not be automated using this model:

- Publishing, rewriting, or deleting content without human review.
- Assuming that refreshing a page will improve SEO performance.
- Making decisions that rely only on the model score while ignoring editorial or business context.
- Treating the model score as proof that a page is currently declining or will decline in the future.
- Using the model outside the type of content and observed data represented in this dataset.

The model is intended to support prioritization, not to replace human decision-making.

## 4. Monitoring / retrain triggers

### Monitoring

The ranked recommendations should be monitored over time to ensure that they remain useful for prioritizing content review. If the characteristics of new content or search behavior change, the model may become less representative of current conditions.

Model performance should also be monitored on newly observed data using the same evaluation metrics, such as Precision@20 and Precision@50, to check whether ranking quality has declined.

### Retrain triggers

The model should be reviewed or retrained if:

- Precision@20 or Precision@50 decreases consistently on new evaluation data.
- A large amount of new content becomes available that was not represented in the original training dataset.
- Search behavior, content strategy, or business priorities change substantially.
- New features become available that better describe page performance.
- The data collection process or feature definitions change.

Retraining should use the same leakage checks and validation approach before the updated model is deployed.

## 5. Exports for the paper



In [20]:
output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

queue_path = os.path.join(
    output_dir,
    "content_action_queue.csv"
)

action_queue[
    [
        "content_id",
        "model_score",
        "recommended_action",
        "reason_codes",
        "human_review_required",
    ]
].to_csv(queue_path, index=False)

print("Saved:", queue_path)
print("Rows exported:", len(action_queue))

Saved: work/outputs/content_action_queue.csv
Rows exported: 6163


In [21]:
exported_queue = pd.read_csv(queue_path)

print("Exported queue shape:", exported_queue.shape)
print("\nColumns:")
print(exported_queue.columns.tolist())

print("\nTop 10 rows:")
display(exported_queue.head(10))

Exported queue shape: (6163, 5)

Columns:
['content_id', 'model_score', 'recommended_action', 'reason_codes', 'human_review_required']

Top 10 rows:


,content_id,model_score,recommended_action,reason_codes,human_review_required
0,content_2ba626fea4d6,0.990,Prioritize refresh review,OLDER_CONTENT,Yes
1,content_e988c1699454,0.975,Prioritize refresh review,"OLDER_CONTENT, LOW_SEARCH_POSITION, HIGH_SEARC...",Yes
2,content_7766ffacdcfa,0.970,Prioritize refresh review,"OLDER_CONTENT, HIGH_SEARCH_VISIBILITY",Yes
3,content_9ac61c04930e,0.955,Prioritize refresh review,"OLDER_CONTENT, HIGH_SEARCH_VISIBILITY",Yes
4,content_790cc62d4743,0.955,Prioritize refresh review,"OLDER_CONTENT, LOW_SEARCH_POSITION, HIGH_SEARC...",Yes
5,content_b23d650634c6,0.950,Prioritize refresh review,MIXED_SIGNALS,Yes
6,content_569475b335ab,0.950,Prioritize refresh review,"OLDER_CONTENT, LOW_SEARCH_POSITION, HIGH_SEARC...",Yes
7,content_0d7f4fecb391,0.940,Prioritize refresh review,"OLDER_CONTENT, LOW_SEARCH_POSITION, HIGH_SEARC...",Yes
8,content_500bd3907331,0.940,Prioritize refresh review,HIGH_SEARCH_VISIBILITY,Yes
9,content_53285c7434ac,0.935,Prioritize refresh review,MIXED_SIGNALS,Yes


### Exported outputs

The ranked content action queue was exported to `work/outputs/content_action_queue.csv`.

The export contains the content identifier, model score, recommended action, reason codes, and human-review requirement. The file is intended to provide a reusable input for the paper and for later review of the model's recommendations.

The queue is sorted by model score so that pages with higher observed similarity to the declining class appear earlier in the review order.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.